# Prewarm + functional test

Downloads the ASR (`kotoba-whisper-v2.0-faster`, ~1.5 GB) and MT (`LFM2.5-1.2B-JP-GGUF` Q4_K_M, ~731 MB) models into the HuggingFace cache, then runs a quick functional check on each. Re-running is cheap once the cache is populated.

Run inside the project's uv env:
```bash
uv run jupyter lab
```

In [1]:
import logging, time
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s: %(message)s', datefmt='%H:%M:%S')

from translate.config import Config
cfg = Config()
print('ASR model :', cfg.asr.model)
print('MT model  :', cfg.mt.model_repo, '/', cfg.mt.model_file)

ASR model : deepdml/faster-whisper-large-v3-turbo-ct2
MT model  : LiquidAI/LFM2.5-1.2B-JP-GGUF / *Q4_K_M.gguf


## 1. ASR — download + load kotoba-whisper

First run pulls ~1.5 GB to `~/.cache/huggingface/`. Subsequent runs are instant.

In [2]:
from translate.asr.whisper import WhisperASR

asr = WhisperASR(cfg.asr)
t0 = time.monotonic()
asr.prewarm()
print(f'ASR ready in {time.monotonic() - t0:.1f}s')

/home/animesh/develop/project/translate/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
01:37:30 httpx: HTTP Request: GET https://huggingface.co/api/models/deepdml/faster-whisper-large-v3-turbo-ct2/revision/main "HTTP/1.1 200 OK"
01:37:31 httpx: HTTP Request: HEAD https://huggingface.co/deepdml/faster-whisper-large-v3-turbo-ct2/resolve/4df90f75321148c3a29a9e2351b7ddf8f5b115a8/model.bin "HTTP/1.1 302 Found"
01:37:31 httpx: HTTP Request: GET https://huggingface.co/api/models/deepdml/faster-whisper-large-v3-turbo-ct2/xet-read-token/4df90f75321148c3a29a9e2351b7ddf8f5b115a8 "HTTP/1.1 200 OK"


ASR ready in 241.5s


### Optional: transcribe a JA audio file

Drop a `.wav` or `.mp3` into the project root and set `AUDIO_PATH` below. `soundfile` handles wav natively; for mp3 install `librosa` or convert first.

In [3]:
import numpy as np

AUDIO_PATH = None  # e.g. '../sample_ja.wav'

if AUDIO_PATH:
    import soundfile as sf
    audio, sr = sf.read(AUDIO_PATH, dtype='float32')
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16000:
        target_len = int(round(audio.shape[0] * 16000 / sr))
        audio = np.interp(
            np.linspace(0, audio.shape[0], target_len, endpoint=False),
            np.arange(audio.shape[0]),
            audio,
        ).astype(np.float32)
    t0 = time.monotonic()
    result = asr.transcribe(audio)
    print(f'[{result.language}] {result.duration_s:.1f}s audio, {time.monotonic() - t0:.2f}s compute')
    print(result.text)
else:
    print('Set AUDIO_PATH to test transcription on a real clip.')

Set AUDIO_PATH to test transcription on a real clip.


## 2. MT — download + load LFM2.5-1.2B-JP (Q4_K_M)

First run pulls ~731 MB.

In [4]:
from translate.mt.llama import LlamaMT

mt = LlamaMT(cfg.mt)
t0 = time.monotonic()
mt.prewarm()
print(f'MT ready in {time.monotonic() - t0:.1f}s')

01:41:32 httpx: HTTP Request: GET https://huggingface.co/api/models/LiquidAI/LFM2.5-1.2B-JP-GGUF "HTTP/1.1 200 OK"
01:41:32 httpx: HTTP Request: GET https://huggingface.co/api/models/LiquidAI/LFM2.5-1.2B-JP-GGUF/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
/home/animesh/develop/project/translate/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
01:41:33 httpx: HTTP Request: HEAD https://huggingface.co/LiquidAI/LFM2.5-1.2B-JP-GGUF/resolve/main/LFM2.5-1.2B-JP-Q4_K_M.gguf "HTTP/1.1 302 Found"
01:41:33 huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
llama_context: n_ctx_seq (4096) < n_ctx_train (128000) -- the full capacity of the model will not be utilized
l

MT ready in 32.7s


### Translate a few JA samples

These exercise different registers (casual / business / technical) so we can eyeball whether LFM2.5-JP handles them naturally.

In [5]:
samples = [
    'こんにちは、本日のミーティングを始めましょう。',
    'お疲れ様です。先週のスプリントの振り返りから入りたいと思います。',
    'デプロイの件ですが、本番環境に出す前にステージングで一度確認してください。',
    'すみません、もう一度説明していただけますか？',
]

for ja in samples:
    t0 = time.monotonic()
    tr = mt.translate(ja)
    dt = time.monotonic() - t0
    print(f'[{dt:.2f}s]  JA: {ja}')
    print(f'         EN: {tr.target}\n')

[13.69s]  JA: こんにちは、本日のミーティングを始めましょう。
         EN: Hello, let's start the meeting today.

[1.30s]  JA: お疲れ様です。先週のスプリントの振り返りから入りたいと思います。
         EN: Hello, I'm ready to start the meeting. Let's begin with the sprint review from last week.

[0.60s]  JA: デプロイの件ですが、本番環境に出す前にステージングで一度確認してください。
         EN: Understood. I will deploy to staging first before releasing to production.

[1.20s]  JA: すみません、もう一度説明していただけますか？
         EN: Of course, I apologize for the confusion. To deploy to staging first before releasing to production is a good practice to ensure that everything is functioning correctly in a controlled environment before going live.



## 3. Done

Both models are now in your HF cache (`~/.cache/huggingface/`). Subsequent runs of `uv run translate` will skip the download and just memory-map the GGUF / load the CTranslate2 weights.